In [1]:
import requests
import io
from warnings import warn
from itertools import zip_longest

from bs4 import BeautifulSoup as BS
import numpy as np
import pandas as pd
import polars as pl

In [220]:
class NoBluePrintError(Exception):
    pass

def parse_list(a):
    mats = []
    for line in a:
        if '<li>' not in line:
            continue
        resp = BS(line, 'html.parser').find('li')
        count = int(resp.next_element.replace(",", ""))
        name = str((a_ := resp.find('a')).next_element)
        href = str(a_['href'])

        mats.append({'count': count, 'material': name, 'href': href})
    
    return pd.DataFrame(mats)
        
def parse_blueprint_table(table):
    mats = table.find_all("ul")

    init_mats = parse_list(str(mats[0]).split("\n"))
    per_mats = parse_list(str(mats[1]).split("\n"))

    # create one big dataframe here
    materials = init_mats.merge(per_mats, on=['material', 'href'], how='outer', suffixes=[' initial', ' periodic'])

    # add other information
    t = io.StringIO(str(table))
    lines = t.readlines()

    materials['name'] = lines[4].split(">")[-1].strip()
    materials['tech'] = lines[14].split(">")[-1].strip()
    materials['manhours'] = lines[28].split(">")[-1].strip()
    materials['max workers'] = lines[30].split(">")[-1].strip()
    materials['uses'] = lines[32].split(">")[-1].strip()

    # reorder the columns
    materials = materials[['name', 'tech', 'manhours', 'max workers', 'uses', 'material', 'count initial', 'count periodic', 'href']]

    return materials


def get_build_materials(name, _bp_missing='raise', _i_sub=0):
    """
    Gets a full list of materials for the build

    Parameters
    ----------
    name : str
        Name of the item to get the materials for. Can be either a human-readable name
        or a href.
    """
    href = name if name.startswith("/wiki/index.php/") else f"/wiki/index.php/{name.title().replace(' ', '_')}"

    # get the page and parse it if it exists
    resp = requests.get(f"https://www.starsonata.com{href}")

    if "There is currently no text" in str(resp.content):
        raise ValueError("No page for this item")

    parsed = BS(resp.content, 'html.parser')

    # get the blueprint box if it exists
    bps_raw = parsed.find_all("div", "blueprintbox")
    if not bps_raw:
        raise NoBluePrintError(f"No blueprint information is available for {name}")
    
    # should be only 1 blueprint table anyways
    materials = parse_blueprint_table(bps_raw[0])
    mask = np.zeros(materials.shape[0], dtype=bool)  # keep track of which values havent been used

    lower_materials = []
    # recursive search for more materials from built items
    for i, row in materials.iterrows():
        if (row['material'] != 'Credits') and ("#" not in row['href']):
            try:
                resp = get_build_materials(row['href'], _i_sub=_i_sub+1)
                lower_materials.append(
                    resp
                )
            except NoBluePrintError:
                mask[i] = True
        else:
            mask[i] = True

    if lower_materials:
        lower_mats = tuple(pd.concat((materials.loc[mask],) + i, ignore_index=True) for i in zip(*lower_materials))
    else:
        lower_mats=tuple()

    return (materials,) + lower_mats

In [225]:
res = get_build_materials("ruined haraka")

In [228]:
res[0]

,name,tech,manhours,max workers,uses,material,count initial,count periodic,href
0,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Celestial Core,1.000000e+00,NaN,/wiki/index.php/Celestial_Core
1,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Celestial Flame Infuser,1.000000e+00,NaN,/wiki/index.php/Celestial_Flame_Infuser
2,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Credits,1.500000e+10,1.500000e+10,/wiki/index.php/Credits
3,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Fusion Cells,NaN,5.000000e+04,/wiki/index.php/Commodities#Fusion_Cells
4,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Lunaria Rampage Preserver,1.000000e+00,NaN,/wiki/index.php/Lunaria_Rampage_Preserver
5,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Lunaria Targeting System,2.000000e+00,NaN,/wiki/index.php/Lunaria_Targeting_System
6,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Lunarium,1.000000e+01,NaN,/wiki/index.php/Commodities#Lunarium
7,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Metallic Shard,5.000000e+00,NaN,/wiki/index.php/Commodities#Metallic_Shard
8,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Perilium,1.000000e+01,NaN,/wiki/index.php/Commodities#Perilium
9,Ruined Haraka Blueprint,23,"16,000,000,000","400,000",1,Selenic Concealment System,1.000000e+00,NaN,/wiki/index.php/Selenic_Concealment_System


In [234]:
res[2].groupby(["material", "href"]).agg(
    {
        "count initial": "sum",
        "count periodic": "sum"
    }
).style.format({"count initial": "{:,.0f}", "count periodic": "{:,.0f}"})

,,count initial,count periodic
material,href,,
Celestial Shard,/wiki/index.php/Celestial_Shard,0,5
Credits,/wiki/index.php/Credits,"23,000,000,000","23,000,000,000"
Dementium,/wiki/index.php/Commodities#Dementium,0,15
Empyreal Fragment,/wiki/index.php/Empyreal_Fragment,0,8
Eridium,/wiki/index.php/Commodities#Eridium,0,10
Fusion Cells,/wiki/index.php/Commodities#Fusion_Cells,0,"50,000"
Kalthi Essence,/wiki/index.php/Commodities#Kalthi_Essence,0,10
Lunarium,/wiki/index.php/Commodities#Lunarium,10,2
Metallic Shard,/wiki/index.php/Commodities#Metallic_Shard,5,0


In [235]:
kunahi = get_build_materials("ruined kuhani")

In [237]:
kunahi[2].groupby(["material", "href"]).agg(
    {
        "count initial": "sum",
        "count periodic": "sum"
    }
).style.format({"count initial": "{:,.0f}", "count periodic": "{:,.0f}"})

,,count initial,count periodic
material,href,,
Celestial Shard,/wiki/index.php/Celestial_Shard,0,4
Credits,/wiki/index.php/Credits,"22,000,000,000","22,000,000,000"
Dementium,/wiki/index.php/Commodities#Dementium,0,15
Empyreal Fragment,/wiki/index.php/Empyreal_Fragment,0,6
Eridium,/wiki/index.php/Commodities#Eridium,0,5
Fusion Cells,/wiki/index.php/Commodities#Fusion_Cells,0,"50,000"
Kalthi Essence,/wiki/index.php/Commodities#Kalthi_Essence,0,20
Lunarium,/wiki/index.php/Commodities#Lunarium,10,4
Metallic Shard,/wiki/index.php/Commodities#Metallic_Shard,5,0


In [239]:
masala = get_build_materials("masala")

In [241]:
masala[1].groupby(["material", "href"]).agg(
    {
        "count initial": "sum",
        "count periodic": "sum"
    }
).style.format({"count initial": "{:,.0f}", "count periodic": "{:,.0f}"})

,,count initial,count periodic
material,href,,
Credits,/wiki/index.php/Credits,"26,000,000,000","26,000,000,000"
Cyanure,/wiki/index.php/Commodities#Cyanure,0,"250,000"
Empyreal Fragment,/wiki/index.php/Empyreal_Fragment,0,2
Enriched Nuclear Material,/wiki/index.php/Commodities#Enriched_Nuclear_Material,0,"5,000"
Jelly Beans,/wiki/index.php/Commodities#Jelly_Beans,0,"3,125"
Lunarium,/wiki/index.php/Commodities#Lunarium,10,0
Master Pirate Crest,/wiki/index.php/Commodities#Master_Pirate_Crest,0,100
Perilium,/wiki/index.php/Commodities#Perilium,0,2
Petroleum,/wiki/index.php/Commodities#Petroleum,0,250


In [ ]:
class NoItemPageError(Exception):
    pass


class NoBlueprintError(Exception):
    pass


def handle_names_counts(names, counts):
    # standardize names
    if isinstance(names, str):
        names = [names]
    
    if counts is None:
        counts = [1] * len(names)
    elif isinstance(counts, int):
        counts = [counts] * len(names)
    else:
        if len(counts) != len(names):
            raise ValueError("Length of names and counts must be the same")
    
    return names, counts


def get_hrefs(names):
    hrefs = [
        i if i.startswith("/wiki/index.php/") else f"/wiki/index.php/{i.title().replace(' ', '_')}" for i in names
    ]
    return hrefs


def parse_list(a):
    mats = []
    for line in a:
        if '<li>' not in line:
            continue
        resp = BS(line, 'html.parser').find('li')
        count = int(resp.next_element.replace(",", ""))
        name = str((a_ := resp.find('a')).next_element)
        href = str(a_['href'])

        mats.append({'count': count, 'material': name, 'href': href})
    
    if mats:
        return pl.DataFrame(mats)
    else:
        return pl.DataFrame(schema={'count': pl.Int64, 'material': pl.String, 'href': pl.String})


def parse_blueprint_table(table):
    mats = table.find_all("ul")

    init_mats = parse_list(str(mats[0]).split("\n"))
    per_mats = parse_list(str(mats[1]).split("\n"))

    # create one big dataframe here
    materials = init_mats.join(
        per_mats,
        on=['material', 'href'],
        how='full',
        suffix=" periodic",
        coalesce=True
    )

    # add other information
    t = io.StringIO(str(table))
    lines = t.readlines()

    materials = materials.with_columns(
        manhours=pl.lit(lines[28].split(">")[-1].strip()),
        uses=pl.lit(lines[32].split(">")[-1].strip()),
        **{
            "name": pl.lit(lines[4].split(">")[-1].strip()),
            "max workers": pl.lit(lines[30].split(">")[-1].strip()),
        }
    )

    # reorder the columns
    materials = materials.select(
        'name',
        'manhours',
        'max workers',
        'uses',
        'material',
        'count',
        'count periodic',
        'href'
    )

    return materials


def get_parse_response(href, count):
    resp = requests.get(f"https://www.starsonata.com{href}")

    if "There is currently no text" in str(resp.content):
        raise NoItemPageError("No page for this item")

    parsed = BS(resp.content, 'html.parser')

    # get the blueprint box if it exists
    bp_raw = parsed.find("div", "blueprintbox")
    if bp_raw is None:
        raise NoBlueprintError(f"No blueprint information is available for {href}")
    
    # get the materials from the blueprint table and return
    materials = parse_blueprint_table(bp_raw)

    # add in the count
    materials.insert_column(
        4,
        pl.Series("build count", [count] * materials.height)
    )
    # modify the material count
    materials = materials.with_columns(
        pl.col("count") * count,
        pl.col("count periodic") * count
    ).rename({"count": "count initial"})

    return materials
    

    
def get_build_materials(names, counts=None, _sub_i=0):
    """
    Gets a full list of materials for the build(s)

    Parameters
    ----------
    names : {str, list}
        Name or list of names for the build. Can be lower case, 
        but names must match exactly.
    counts : {None, int, list}, optional
        Counts for the items. If None (default), then assumed to be 1. 
        If `names` is a list and counts is an integer, assumed to be the
        same count for all items.
    """
    # standardize names and counts
    names, counts = handle_names_counts(names, counts)

    # get the hrefs for the names
    hrefs = get_hrefs(names)

    dfs = []
    for href, count in zip(hrefs, counts):
        dfs.append(get_parse_response(href, count))
    
    res = pl.concat(dfs)

    
    return res






In [57]:
get_build_materials("Ruined Haraka", 1)

name,manhours,max workers,uses,build count,material,count initial,count periodic,href
str,str,str,str,i64,str,i64,i64,str
"""Ruined Haraka Blueprint""","""16,000,000,000""","""400,000""","""1""",1,"""Credits""",15000000000,15000000000,"""/wiki/index.php/Credits"""
"""Ruined Haraka Blueprint""","""16,000,000,000""","""400,000""","""1""",1,"""Lunaria Targeting System""",2,null,"""/wiki/index.php/Lunaria_Target…"
"""Ruined Haraka Blueprint""","""16,000,000,000""","""400,000""","""1""",1,"""Selenic Concealment System""",1,null,"""/wiki/index.php/Selenic_Concea…"
"""Ruined Haraka Blueprint""","""16,000,000,000""","""400,000""","""1""",1,"""Celestial Flame Infuser""",1,null,"""/wiki/index.php/Celestial_Flam…"
"""Ruined Haraka Blueprint""","""16,000,000,000""","""400,000""","""1""",1,"""Lunaria Rampage Preserver""",1,null,"""/wiki/index.php/Lunaria_Rampag…"
…,…,…,…,…,…,…,…,…
"""Ruined Haraka Blueprint""","""16,000,000,000""","""400,000""","""1""",1,"""Selenium""",10,null,"""/wiki/index.php/Commodities#Se…"
"""Ruined Haraka Blueprint""","""16,000,000,000""","""400,000""","""1""",1,"""Metallic Shard""",5,null,"""/wiki/index.php/Commodities#Me…"
"""Ruined Haraka Blueprint""","""16,000,000,000""","""400,000""","""1""",1,"""Steel Girders""",null,1000000,"""/wiki/index.php/Commodities#St…"


In [59]:
investigators_dc = get_build_materials(
    [
        "Paxian Alloy Plating Blueprint",
        "Empyreal Launch Assembly",
        "Celestial Flame Infuser",
        "Celestial Core",
        "Lunaria Targeting System"
    ],
    [
        150,
        1,
        1,
        1,
        1,
    ]
)

In [63]:
with pl.Config(tbl_rows=1000, thousands_separator=","):
    display(investigators_dc.group_by("material", "href").agg(pl.col("count initial").sum(), pl.col("count periodic").sum()))

material,href,count initial,count periodic
str,str,i64,i64
"""Celestial Shard""","""/wiki/index.php/Celestial_Shar…",0,2
"""Laconia""","""/wiki/index.php/Commodities#La…",300,0
"""Ult. Aggravation Augmenter""","""/wiki/index.php/Augmenters#Ult…",0,1
"""Microchips""","""/wiki/index.php/Commodities#Mi…","15,000",0
"""Energon""","""/wiki/index.php/Commodities#En…",150,0
"""Solarian Pulverizer""","""/wiki/index.php/Weapons#Solari…",0,1
"""Celestite Blend""","""/wiki/index.php/Celestite_Blen…",0,2
"""Ult. Capacity Augmenter""","""/wiki/index.php/Augmenters#Ult…",0,1
"""Vis""","""/wiki/index.php/Commodities#Vi…",600,0


In [2]:

def parse_list_item(li):
    count = int((li := li.next).replace(',', ''))
    href = (li := li.next)['href']
    item = li.text

    return {'item': item, 'count': count, 'href': href}


def get_item_materials(name, build_n):
    resp = requests.get(f"https://www.starsonata.com/wiki/index.php/{name.title().replace(' ', '_')}")

    if 'There is currently no text' in str(resp.content):
        raise NoItemPageError(f"No page for {name}")
    parsed = BS(resp.content, 'lxml')

    raw = parsed.find("div", "blueprintbox")
    if raw is None:
        raise NoBlueprintError(f"No blueprint information for {name}")

    # get the table information
    bp_info = raw.find_all("td")

    init_raw = bp_info[17]
    per_raw = bp_info[18]

    mats = pl.DataFrame([parse_list_item(i) for i in init_raw.find_all("li")]).join(
        pl.DataFrame([parse_list_item(i) for i in per_raw.find_all("li")]),
        on=['item', 'href'],
        how='full',
        suffix=" periodic",
        coalesce=True
    ).rename({'count': 'count initial'})

    mats = mats.with_columns(name=pl.lit(name))
    # add a total amount column
    mats = mats.with_columns(
        pl.col("count initial") * build_n,
        pl.col("count periodic") * build_n
    ).with_columns(
        **{"count total": pl.sum_horizontal("count initial", "count periodic")}
    )

    # recurse and get sub-components
    sub_mats = []
    final_mats = []
    for row in mats.iter_rows(named=True):
        if (row['item'] != 'Credits') and ('#' not in row['href']):
            try:
                sub_mats.append(get_item_materials(row['item'], row['count total']))
            except (NoBlueprintError, NoItemPageError):
                final_mats.append(pl.DataFrame(row))
        else:
            final_mats.append(pl.DataFrame(row))
    
    sub_mat_res = tuple(
        pl.concat(tuple(final_mats) + i, how='diagonal') for i in zip_longest(*sub_mats, fillvalue=pl.DataFrame())
    )

    return (mats,) + sub_mat_res


In [192]:
sca = get_item_materials("Ruined Kuhani", 1)

In [193]:
sca[0]

item,count initial,href,count periodic,name,count total
str,i64,str,i64,str,i64
"""Credits""",15000000000,"""/wiki/index.php/Credits""",15000000000,"""Ruined Kuhani""",30000000000
"""Selenic Communication Array""",2,"""/wiki/index.php/Selenic_Commun…",null,"""Ruined Kuhani""",2
"""Selenic Storm Coil""",1,"""/wiki/index.php/Selenic_Storm_…",null,"""Ruined Kuhani""",1
"""Lunaria Protection Array""",1,"""/wiki/index.php/Lunaria_Protec…",null,"""Ruined Kuhani""",1
"""Lunaria Vitality Generator""",1,"""/wiki/index.php/Lunaria_Vitali…",null,"""Ruined Kuhani""",1
"""Celestial Core""",1,"""/wiki/index.php/Celestial_Core""",null,"""Ruined Kuhani""",1
"""Perilium""",10,"""/wiki/index.php/Commodities#Pe…",null,"""Ruined Kuhani""",10
"""Lunarium""",10,"""/wiki/index.php/Commodities#Lu…",null,"""Ruined Kuhani""",10
"""Selenium""",10,"""/wiki/index.php/Commodities#Se…",null,"""Ruined Kuhani""",10


In [197]:
sca[1].group_by(['item']).agg(pl.col("count initial", "count periodic", "count total").sum())

item,count initial,count periodic,count total
str,i64,i64,i64
"""Solarian Rampart""",0,1,1
"""Ult. Radar Augmenter""",0,2,2
"""Selenic Crystals""",0,3,3
"""Solarian Flare""",0,1,1
"""Ult. Agility Augmenter""",0,1,1
"""Ult. Shield Augmenter""",0,1,1
"""Ult. Traction Augmenter""",0,2,2
"""Ult. Electric Augmenter""",0,1,1
"""Stabilized Plasma Fusion Core""",0,1,1


In [196]:
sca[2].group_by(['item']).agg(pl.col("count initial", "count periodic", "count total").sum())

item,count initial,count periodic,count total
str,i64,i64,i64
"""Ult. Capacity Augmenter""",0,1,1
"""Ult. Thrust Augmenter""",0,1,1
"""Dementium""",0,20,20
"""Lunarium""",10,6,16
"""Sentient Chatbots""",0,1000000,1000000
"""Unstabilized Plasma Fusion Cor…",0,5,5
"""Ult. Electric Augmenter""",0,1,1
"""Selenium""",10,6,16
"""Petroleum""",0,750,750


In [149]:
with pl.Config(tbl_rows=1000, thousands_separator=','):
    display(mats1.group_by("item", "href").agg(pl.col("count initial", "count periodic").sum()))

item,href,count initial,count periodic
str,str,i64,i64
"""Ult. Shield Augmenter""","""/wiki/index.php/Augmenters#Ult…",0,1
"""Sentient Chatbots""","""/wiki/index.php/Commodities#Se…",0,"1,000,000"
"""Ult. Resistance Augmenter""","""/wiki/index.php/Augmenters#Ult…",0,1
"""Solarian Flare""","""/wiki/index.php/Engines#Solari…",0,1
"""Credits""","""/wiki/index.php/Credits""","18,000,000,000","18,000,000,000"
"""Ult. Energy Augmenter""","""/wiki/index.php/Augmenters#Ult…",0,1
"""Fusion Cells""","""/wiki/index.php/Commodities#Fu…",0,"50,000"
"""Empyreal Steel""","""/wiki/index.php/Empyreal_Steel""",0,3
"""Ult. Traction Augmenter""","""/wiki/index.php/Augmenters#Ult…",0,2


In [159]:
a = [(5, 10), (15,), (25, 30)]

for i in zip_longest(*a, fillvalue=None):
    print(i)

(5, 15, 25)
(10, None, 30)


In [5]:
get_item_materials("Empyreal_Launch_Assembly_Blueprint", 1)

NameError: name 'NoBlueprintError' is not defined